# DeepGuard : entraînement sur Kaggle

**Avant de lancer :**
1. *Session options* : accélérateur **GPU** (T4 ou P100) et **Internet On**.
2. *Add Input* : ajoute le dataset **140k Real and Fake Faces** (`xhlulu/140k-real-and-fake-faces`).
   Pour le test de généralisation, ajoute aussi **Real and Fake Face Detection** (`ciplab/real-and-fake-face-detection`).
3. *Add-ons → Secrets* : crée `DAGSHUB_TOKEN` (token DagsHub) pour suivre les runs dans MLflow. Sans lui, les runs restent locaux.
4. Le dépôt GitHub doit être **public** pour le `git clone` ci-dessous (sinon utilise un token).

Exécute les cellules dans l'ordre. Commence avec les petites limites de `params.yaml` (20 000 images), puis passe à `0` (tout le dataset) pour le modèle retenu.

In [ ]:
# 1. Récupérer le code et installer le nécessaire (PyTorch, sklearn, pandas, matplotlib existent déjà sur Kaggle)
!git clone https://github.com/sourourbensalha209-alt/deepguard.git
%cd deepguard
!pip install -q timm mlflow pyyaml onnx onnxruntime

In [ ]:
# 2. Chemins des données + suivi MLflow (DagsHub)
import glob, os, yaml

cands = glob.glob("/kaggle/input/**/real-vs-fake", recursive=True)
assert cands, "Dataset introuvable : ajoute 140k Real and Fake Faces via « Add Input »."
os.environ["DATA_DIR"] = cands[0]
print("DATA_DIR =", os.environ["DATA_DIR"])

ext = [p for p in glob.glob("/kaggle/input/**/training_fake", recursive=True)]
EXTERNAL_DIR = os.path.dirname(ext[0]) if ext else None
print("external_dir =", EXTERNAL_DIR)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/sourourbensalha209-alt/deepguard.mlflow"
    os.environ["MLFLOW_TRACKING_USERNAME"] = "sourourbensalha209-alt"
    os.environ["MLFLOW_TRACKING_PASSWORD"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
    print("MLflow : DagsHub")
except Exception as e:
    os.environ.pop("MLFLOW_TRACKING_URI", None)
    print("MLflow : local (secret DAGSHUB_TOKEN absent) :", e)

def set_params(external=None, **train):
    p = yaml.safe_load(open("params.yaml"))
    p["train"].update(train)
    if external:
        p["external_dir"] = external
    yaml.safe_dump(p, open("params.yaml", "w"), sort_keys=False)
    print(p["train"])

if EXTERNAL_DIR:
    set_params(external=EXTERNAL_DIR)

In [ ]:
# 3. Listes de fichiers train / val / test (équilibrées, reproductibles)
!python -m src.data.prepare

## 4. Comparer les trois modèles
Chaque run apparaît dans MLflow. Les checkpoints sont copiés dans `/kaggle/working`.

In [ ]:
# Baseline CNN (entraîné from scratch)
set_params(model="baseline", epochs=10, batch_size=128, lr=1e-3)
!python -m src.models.train
!cp checkpoints/best.pt /kaggle/working/baseline.pt && cp metrics/train.json /kaggle/working/train_baseline.json

In [ ]:
# EfficientNet-V2-S (transfer learning)
set_params(model="efficientnet", epochs=3, batch_size=64, lr=1e-4)
!python -m src.models.train
!cp checkpoints/best.pt /kaggle/working/efficientnet.pt && cp metrics/train.json /kaggle/working/train_efficientnet.json

In [ ]:
# ViT-B/16 (Vision Transformer)
set_params(model="vit", epochs=3, batch_size=32, lr=3e-5)
!python -m src.models.train
!cp checkpoints/best.pt /kaggle/working/vit.pt && cp metrics/train.json /kaggle/working/train_vit.json

In [ ]:
# Tableau comparatif sur la validation
import json, pandas as pd
rows = [json.load(open(f"/kaggle/working/train_{m}.json")) for m in ["baseline", "efficientnet", "vit"]
        if os.path.exists(f"/kaggle/working/train_{m}.json")]
pd.DataFrame(rows)[["model", "val_accuracy", "val_f1", "val_roc_auc", "train_time_s", "n_params_m"]]

## 5. Évaluer, expliquer et exporter le modèle retenu
Choisis le gagnant (`MODEL`). L'export ONNX embarque l'explication : **CAM** pour un CNN, **Attention Rollout** pour le ViT. L'API et le frontend affichent donc la carte dans les deux cas.

In [ ]:
MODEL = "efficientnet"   # baseline | efficientnet | vit

hp = {"baseline": dict(epochs=10, batch_size=128, lr=1e-3),
      "efficientnet": dict(epochs=3, batch_size=64, lr=1e-4),
      "vit": dict(epochs=3, batch_size=32, lr=3e-5)}[MODEL]
set_params(model=MODEL, **hp)
!cp /kaggle/working/{MODEL}.pt checkpoints/best.pt
!cp /kaggle/working/train_{MODEL}.json metrics/train.json
!python -m src.models.evaluate
!python -m src.utils.gradcam

In [ ]:
# Export ONNX : logits + carte d'explication (CAM pour un CNN, Attention Rollout pour le ViT),
# parité PyTorch/ONNX, latence et quantification INT8
!python -m src.models.export_onnx

In [ ]:
# Tout ce qu'il faut récupérer, dans un seul zip (onglet Output du notebook)
!zip -r /kaggle/working/deepguard_outputs.zip checkpoints/best.pt data/processed models metrics reports params.yaml
!ls -lh /kaggle/working/deepguard_outputs.zip

## 6. De retour sur ton ordinateur
Dézippe `deepguard_outputs.zip` à la racine du dépôt, puis :
```bash
dvc commit -f        # enregistre les sorties (modèle, rapports...) comme résultat du pipeline : crée dvc.lock
dvc push             # envoie modèle et rapports sur DagsHub
git add dvc.lock params.yaml metrics && git commit -m "feat: modèle entraîné" && git push
docker compose up --build   # l'API détecte models/deepguard.onnx et remplace le modèle de démo
```